In [ ]:
from pymilvus import MilvusClient, Collection,connections, db, DataType

In [ ]:
import pymilvus
print(pymilvus.__version__)

In [ ]:
MILVUS_HOST="10.18.27.59"
MILVUS_PORT="19530"
conn = connections.connect( 
            host= MILVUS_HOST,
            post = MILVUS_PORT
        )
print(db.list_database())

db_name = "test_db"
if db_name not in db.list_database():
    database = db.create_database(db_name)

In [ ]:
client = MilvusClient(
        host= MILVUS_HOST,
        port=  MILVUS_PORT,
        db_name = "test_db"
    )
lst_db = client.list_databases() # in version 2.5.x you can list db, connect milvus without use module connection, db
lst_collection = client.list_collections()
print(lst_db)
print(lst_collection)

In [ ]:
collection_name = "test"
if collection_name not in client.list_collections():
    # Create schema
    schema = MilvusClient.create_schema(
        auto_id=True,
        enable_dynamic_field=True,
    )

    # Add fields to schema
    schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True)
    schema.add_field(field_name="text_vector", datatype=DataType.FLOAT_VECTOR, dim=5)
    schema.add_field(field_name="source", datatype=DataType.VARCHAR, max_length=200)
    # schema.add_field(field_name="lst_img_vector", datatype=DataType.ARRAY, element_type=DataType.FLOAT_VECTOR, dim = 6 ,max_capacity=20)
    schema.add_field(field_name="img_vector", datatype=DataType.FLOAT_VECTOR, dim=5)

    index_params = client.prepare_index_params()

    index_params.add_index(
        field_name="id",
        index_type="AUTOINDEX"
    )
    index_params.add_index(
        field_name="text_vector",
        metric_type="COSINE",
        index_type="IVF_FLAT",
        index_name="vector_text_index",
        params={ "nlist": 128 }
    )
    index_params.add_index(
        # field_name="lst_img_vector",
        field_name="img_vector",
        metric_type="COSINE",
        index_type="IVF_FLAT",
        index_name="img_vector_index",
        params={ "nlist": 128 }
    )
    client.create_collection(
        collection_name=collection_name,
        schema=schema,
        index_params=index_params
    )
    print("Create collection successfully!")

In [ ]:
data = [ 
    {"text_vector": [0.3580376395471989, -0.6023495712049978, 0.18414012509913835, -0.26286205330961354, 0.9029438446296592], "source": "pink_8682","img_vector":[0.3580376395471989, -0.6023495712049978, 0.18414012509913835, -0.26286205330961354, 0.9029438446296592]},
    {"text_vector": [0.19886812562848388, 0.06023560599112088, 0.6976963061752597, 0.2614474506242501, 0.838729485096104], "source": "red_7025","img_vector":[0.19886812562848388, 0.06023560599112088, 0.6976963061752597, 0.2614474506242501, 0.838729485096104]},
    {"text_vector": [0.43742130801983836, -0.5597502546264526, 0.6457887650909682, 0.7894058910881185, 0.20785793220625592], "source": "orange_6781","img_vector":[0.43742130801983836, -0.5597502546264526, 0.6457887650909682, 0.7894058910881185, 0.20785793220625592]},
    {"text_vector": [0.3172005263489739, 0.9719044792798428, -0.36981146090600725, -0.4860894583077995, 0.95791889146345], "source": "pink_9298","img_vector":[0.3172005263489739, 0.9719044792798428, -0.36981146090600725, -0.4860894583077995, 0.95791889146345]},
    {"text_vector": [0.4452349528804562, -0.8757026943054742, 0.8220779437047674, 0.46406290649483184, 0.30337481143159106], "source": "red_4794","img_vector":[0.4452349528804562, -0.8757026943054742, 0.8220779437047674, 0.46406290649483184, 0.30337481143159106]},
    {"text_vector": [0.985825131989184, -0.8144651566660419, 0.6299267002202009, 0.1206906911183383, -0.1446277761879955], "source": "yellow_4222","img_vector":[0.985825131989184, -0.8144651566660419, 0.6299267002202009, 0.1206906911183383, -0.1446277761879955]},
    {"text_vector": [0.8371977790571115, -0.015764369584852833, -0.31062937026679327, -0.562666951622192, -0.8984947637863987], "source": "red_9392","img_vector":[0.8371977790571115, -0.015764369584852833, -0.31062937026679327, -0.562666951622192, -0.8984947637863987]},
    {"text_vector": [-0.33445148015177995, -0.2567135004164067, 0.8987539745369246, 0.9402995886420709, 0.5378064918413052], "source": "grey_8510","img_vector":[-0.33445148015177995, -0.2567135004164067, 0.8987539745369246, 0.9402995886420709, 0.5378064918413052]},
    {"text_vector": [0.39524717779832685, 0.4000257286739164, -0.5890507376891594, -0.8650502298996872, -0.6140360785406336], "source": "white_9381","img_vector":[0.39524717779832685, 0.4000257286739164, -0.5890507376891594, -0.8650502298996872, -0.6140360785406336]},
    {"text_vector": [0.5718280481994695, 0.24070317428066512, -0.3737913482606834, -0.06726932177492717, -0.6980531615588608], "source": "purple_4976","img_vector":[0.5718280481994695, 0.24070317428066512, -0.3737913482606834, -0.06726932177492717, -0.6980531615588608]}
]

In [ ]:
res = client.insert(
    collection_name="test",
    data=data
)

print(res)

In [ ]:
# Single vector search
query_vector = [0.3580376395471989, -0.6023495712049978, 0.18414012509913835, -0.26286205330961354, 0.9029438446296592]
res = client.search(
    collection_name="test",
    anns_field="img_vector",
    data=[query_vector],
    limit=3,
    search_params={"metric_type": "COSINE"}
)

for hits in res:
    for hit in hits:
        print(hit)